# ЛР № 4. Витрина поездок Green Taxi

**Основной вопрос:** В каких зонах посадки, днях недели и часах различаются число
поездок, медианная длительность и средняя итоговая стоимость поездки?

**Уточняющий вопрос 1:** Как соотносятся расстояние, длительность и итоговая
стоимость поездок в выбранных временных срезах?

**Уточняющий вопрос 2:** В каких сочетаниях зон посадки и высадки наблюдается
наибольшее число поездок?

Продолжаем [Project Brief](../docs/project_brief.md) и
[решения ЛР № 3](../docs/data_quality_report.md). Готовим данные для всех трёх
вопросов; полный расчёт метрик и содержательные выводы остаются следующему
этапу. Срезы: зоны посадки и высадки с названиями и районами из справочника,
дата, ISO-день недели и час посадки. Дополнительная мера - расстояние поездки.
Модель и происхождение полей описаны в [docs/mart.md](../docs/mart.md).

## 1. Параметры и вход

Логическая дата берётся из пути подготовленного `staging`, а не из даты запуска.
Время поездок относится к январю 2025 года. `source_ingested_on=2026-09-30`
обозначает загрузку исходных данных и не используется как время события.

Ноутбук запускается из корня репозитория или каталога `notebooks` с ядром
проектного `.venv`. Стенд должен работать; настройки S3 читаются из `.env`.
Ingestion и ЛР № 3 повторно выполнять не нужно.

Справочник уже загружен на стенд в `raw`, отдельного объекта справочника в `staging`
нет. Читаем существующий CSV-снимок за `2026-09-30` напрямую из S3, проверяем его
и включаем названия в широкую таблицу. Повторная загрузка и запись справочника
в `staging` для этого демонстрационного примера не выполняются.


In [1]:
from datetime import date

DATASET_SLUG = "green_tripdata"
SOURCE_INGESTED_ON = "2026-09-30"
MART_NAME = "trips"
MAX_DISTANCE_FOR_RELATIONSHIP_MILES = 1000
ZONES_INGESTED_ON = "2026-09-30"
date.fromisoformat(SOURCE_INGESTED_ON)
date.fromisoformat(ZONES_INGESTED_ON)

PREFIX = f"{DATASET_SLUG}/source_ingested_on={SOURCE_INGESTED_ON}"
STAGING_PATH = f"s3://staging/{PREFIX}/staging.parquet"
MART_KEY = f"{PREFIX}/{MART_NAME}.parquet"
MART_PATH = f"s3://mart/{MART_KEY}"
ZONES_KEY = f"taxi_zone_lookup/ingested_on={ZONES_INGESTED_ON}/taxi_zone_lookup.csv"
ZONES_PATH = f"s3://raw/{ZONES_KEY}"

print(f"Вход:  {STAGING_PATH}")
print(f"Справочник: {ZONES_PATH}")
print(f"Выход: {MART_PATH}")

Вход:  s3://staging/green_tripdata/source_ingested_on=2026-09-30/staging.parquet
Справочник: s3://raw/taxi_zone_lookup/ingested_on=2026-09-30/taxi_zone_lookup.csv
Выход: s3://mart/green_tripdata/source_ingested_on=2026-09-30/trips.parquet


In [2]:
import os
import sys
from html import escape
from pathlib import Path

import boto3
import duckdb
from dotenv import load_dotenv
from IPython.display import HTML, display

PROJECT_ROOT = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "pyproject.toml").is_file()
    and (parent / "src/analysis/read_raw.py").is_file()
)
load_dotenv(PROJECT_ROOT / ".env", override=True)
required_settings = ("S3_ENDPOINT", "S3_ACCESS_KEY", "S3_SECRET_KEY")
missing_settings = [name for name in required_settings if not os.getenv(name)]
if missing_settings:
    raise RuntimeError(f"Не заданы настройки: {', '.join(missing_settings)}")

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from analysis.read_raw import configure_s3  # noqa: E402


def literal(value):
    return "'" + value.replace("'", "''") + "'"

def show_table(headers, rows):
    header = "".join(f"<th>{escape(str(item))}</th>" for item in headers)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(item))}</td>" for item in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))

check_results = []

def check(label, actual, expected):
    passed = actual == expected
    check_results.append((label, actual, expected, passed))
    assert passed, f"{label}: получено {actual!r}, ожидалось {expected!r}"

connection = duckdb.connect()
configure_s3(connection)
endpoint = os.environ["S3_ENDPOINT"]
client = boto3.client(
    "s3",
    endpoint_url=endpoint if endpoint.startswith("http") else f"http://{endpoint}",
    aws_access_key_id=os.environ["S3_ACCESS_KEY"],
    aws_secret_access_key=os.environ["S3_SECRET_KEY"],
    region_name="us-east-1",
)
source_before = client.head_object(Bucket="staging", Key=f"{PREFIX}/staging.parquet")
zones_before = client.head_object(Bucket="raw", Key=ZONES_KEY)


In [3]:
connection.execute(f"""
    CREATE OR REPLACE TEMP VIEW staging AS
    SELECT * FROM read_parquet({literal(STAGING_PATH)}, hive_partitioning = false)
""")
source_schema = {row[0]: row[1] for row in connection.execute("DESCRIBE staging").fetchall()}
required_fields = {
    "lpep_pickup_datetime", "lpep_dropoff_datetime", "PULocationID", "DOLocationID",
    "trip_distance", "total_amount",
    "quality_negative_total_amount_is_correction_violated",
}
check("Наличие нужных полей staging", required_fields - source_schema.keys(), set())

source_counts = connection.execute("""
    SELECT
        count(*) AS rows_count,
        count(*) FILTER (WHERE total_amount IS NULL) AS null_amount,
        count(*) FILTER (
            WHERE quality_negative_total_amount_is_correction_violated
        ) AS flagged_rows,
        count(*) FILTER (
            WHERE lpep_pickup_datetime IS NULL
               OR lpep_dropoff_datetime IS NULL
               OR lpep_dropoff_datetime <= lpep_pickup_datetime
               OR lpep_pickup_datetime < TIMESTAMP '2025-01-01'
               OR lpep_pickup_datetime >= TIMESTAMP '2025-02-01'
        ) AS critical_violations
    FROM staging
""").fetchone()
staging_rows, source_null_amount, source_flagged_rows, source_critical = source_counts
check("Критические нарушения ЛР № 3 на входе", source_critical, 0)
show_table(
    ["Строк staging", "NULL суммы", "Отрицательные суммы с флагом", "Критические нарушения"],
    [source_counts],
)
show_table(
    ["Нужное поле staging", "Тип"],
    [(name, source_schema[name]) for name in sorted(required_fields)],
)


source_distance_profile = connection.execute("""
    SELECT
        count(*) FILTER (WHERE trip_distance IS NULL),
        count(*) FILTER (WHERE trip_distance = 0),
        count(*) FILTER (WHERE trip_distance < 0),
        count(*) FILTER (WHERE NOT isfinite(trip_distance)),
        min(trip_distance), max(trip_distance)
    FROM staging
""").fetchone()
show_table(["NULL расстояния", "Нулей", "Отрицательных", "Нефинитных", "Минимум", "Максимум"], [
    source_distance_profile
])


Строк staging,NULL суммы,Отрицательные суммы с флагом,Критические нарушения
48235,0,152,0


Нужное поле staging,Тип
DOLocationID,INTEGER
PULocationID,INTEGER
lpep_dropoff_datetime,TIMESTAMP
lpep_pickup_datetime,TIMESTAMP
quality_negative_total_amount_is_correction_violated,BOOLEAN
total_amount,DOUBLE
trip_distance,DOUBLE


NULL расстояния,Нулей,Отрицательных,Нефинитных,Минимум,Максимум
0,2629,0,0,0.0,84731.57


## 2. Гранулярность и модель

**Одна строка `trips` представляет одну сохранённую запись о поездке из выбранного
`staging`.** Надёжного бизнес-ключа нет. Отсутствие полных повторов в ЛР № 3
не доказывает уникальность реальных поездок. Не добавляем технический ключ,
не используем `DISTINCT` и сохраняем кратность совпадающих проекций строк.
Все строки `staging` сохраняются в витрине, включая нулевые и большие расстояния.

Выбираем одну широкую таблицу: коды, названия и районы зон посадки и высадки,
календарные признаки, длительность, расстояние и сумма находятся в одной строке.
Подключаем один справочник в двух ролях через два `LEFT JOIN`:
зона посадки (`pz`) по `staging.PULocationID = pz.LocationID`,
зона высадки (`dz`) по `staging.DOLocationID = dz.LocationID`. Одна строка справочника
описывает один код зоны; его ключ `LocationID` должен быть заполнен и уникален.
Для каждого соединения ожидается связь `N:1`: много записей о поездках к одной зоне.

`LEFT JOIN` сохраняет поездки без соответствия, но сам по себе не защищает от
размножения строк при повторах справа. До сборки проверяем ключ справочника,
количество строк после каждого соединения и число ненайденных соответствий. В этом снимке
ожидаем полное покрытие. При нарушении ожиданий останавливаемся до записи.
Атрибуты справочника включаем в широкую таблицу; отдельную таблицу измерения
в `mart` не создаём.

| Будущая метрика | Правило | Агрегация по срезам |
| --- | --- | --- |
| Число поездок | `count(*)` по всем сохранённым записям, включая помеченные | Числа записей можно складывать для непересекающихся групп одного снимка |
| Медианная длительность | `median(duration_minutes)` в минутах, длительность положительна | Рассчитывать из строк нужной группы; медианы групп не складывать и не усреднять |
| Расстояние для уточняющего вопроса | `trip_distance` в милях; для анализа связи с длительностью и стоимостью только `trip_distance > 0 AND trip_distance <= 1000` | Подходящие значения можно суммировать по непересекающимся группам; расстояние не определяет включение строки в остальные показатели |
| Средняя итоговая стоимость | `avg(total_amount)` в USD только для неотрицательных сумм | Рассчитывать из подходящих строк; средние групп не складывать, объединять с учётом числа подходящих значений |

Длительности и суммы отдельных записей можно суммировать в непересекающихся
группах одного снимка, но сумма длительностей не означает время работы парка,
а итоговая сумма оплаты не является доказанной выручкой сервиса. Разные снимки
могут содержать одни и те же поездки; складывать их без проверки пересечений нельзя.

В ЛР № 3 отрицательные суммы оставлены с флагом. Он не доказывает причину
отрицательного значения. Эти строки нужны для числа записей и длительности;
из будущей средней стоимости исключается только их сумма. Это решение ЛР № 3 сохраняется.


Расстояние переносим без изменения, включая нули и все значения больше 1000.
Для дальнейшего анализа связи расстояния с длительностью и стоимостью строки
с `trip_distance = 0` или `trip_distance > 1000` обрабатываются одинаково:
не участвуют в этом анализе, но сохраняются для расчёта остальных показателей.
В выбранном снимке это 2 629 нулевых расстояний и 21 расстояние больше 1000.

Для будущего анализа связи используем условие
`trip_distance > 0 AND trip_distance <= 1000 AND isfinite(trip_distance)`.
При сравнении со стоимостью дополнительно требуется заполненная неотрицательная
сумма. Число записей, медиана длительности, средняя стоимость с правилом по сумме
и число поездок между зонами рассчитываются с сохранением строк обеих групп.
Ограничение расстояния относится к будущему расчёту и не применяется при сборке витрины.

Единица расстояния - мили согласно
[словарю NYC TLC](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_green.pdf).


<a id="distance-policy"></a>
### Общее правило для нулевых и больших расстояний

Все строки `staging` включаются в витрину. Записи с `trip_distance = 0`
и `trip_distance > 1000` сохраняются без изменения расстояния и учитываются
в остальных показателях по действующим правилам длительности и суммы.
Обе группы одинаково исключаются только из дальнейшего анализа связи
расстояния с длительностью и стоимостью.

Для такого анализа применяется условие
`trip_distance > 0 AND trip_distance <= 1000 AND isfinite(trip_distance)`.
Ровно 1000 миль входит в допустимый диапазон. При сравнении со стоимостью
также используется заполненная неотрицательная сумма. Здесь считаем
контрольные количества групп, не фильтруя строки витрины и не рассчитывая
полный набор аналитических метрик.

In [4]:
source_distance_policy_profile = connection.execute(f"""
    SELECT
        count(*) FILTER (WHERE trip_distance = 0),
        count(*) FILTER (WHERE trip_distance > {MAX_DISTANCE_FOR_RELATIONSHIP_MILES}),
        count(*) FILTER (
            WHERE trip_distance = 0 OR trip_distance > {MAX_DISTANCE_FOR_RELATIONSHIP_MILES}
        ),
        count(*) FILTER (
            WHERE trip_distance > 0
              AND trip_distance <= {MAX_DISTANCE_FOR_RELATIONSHIP_MILES}
              AND isfinite(trip_distance)
        )
    FROM staging
""").fetchone()
show_table(["Нули", "Больше 1000", "Не участвуют в анализе связи", "Подходящие расстояния"], [
    source_distance_policy_profile
])

Нули,Больше 1000,Не участвуют в анализе связи,Подходящие расстояния
2629,21,2650,45585


## 3. SQL-преобразование

Выбираем пятнадцать полей для основного и уточняющих вопросов. Справочник
используется для зон посадки и высадки. Все строки сохраняются;
расстояние переносится без изменения.
Время источника сохраняем без преобразования часового пояса. `pickup_isodow`:
1 - понедельник, 7 - воскресенье. Длительность считаем в секундах с переводом
в минуты, без округления до целых минут. Денежное поле и флаг переносим
без изменения значений и типов.

<a id="join-checks"></a>
### Проверки справочника и двух соединений до записи

Сначала проверяем наличие нужных полей CSV, пропуски и повторы `LocationID`,
заполненность названия зоны и района. Повторы ключа не удаляем через `DISTINCT`:
они требуют отдельного решения. Одна строка справочника должна соответствовать
одному коду зоны. Значения `N/A`, `Unknown` и `Outside of NYC` сохраняем как
в источнике: наличие соответствия по коду не доказывает точность географии.

Затем последовательно проверяем два `LEFT JOIN`: сначала зону посадки,
потом зону высадки. После каждого соединения число строк должно совпасть
с предыдущим этапом, ненайденных соответствий быть не должно. Вместе
с уникальностью ключа это подтверждает кардинальность `N:1` для каждой роли.

In [5]:
connection.execute(f"""
    CREATE OR REPLACE TEMP TABLE taxi_zones AS
    SELECT * FROM read_csv_auto(
        {literal(ZONES_PATH)}, header = true, hive_partitioning = false
    )
""")
lookup_schema = {row[0]: row[1] for row in connection.execute("DESCRIBE taxi_zones").fetchall()}
required_zone_fields = {"LocationID", "Zone", "Borough"}
check("Наличие полей справочника", required_zone_fields - lookup_schema.keys(), set())
show_table(["Поле справочника", "Тип"], sorted(lookup_schema.items()))

lookup_profile = connection.execute("""
    SELECT
        count(*) AS lookup_rows,
        count(*) FILTER (WHERE LocationID IS NULL) AS null_keys,
        (SELECT count(*) FROM (
            SELECT LocationID FROM taxi_zones
            GROUP BY LocationID HAVING count(*) > 1
        )) AS duplicate_keys,
        count(*) FILTER (
            WHERE Zone IS NULL OR trim(Zone) = ''
               OR Borough IS NULL OR trim(Borough) = ''
        ) AS missing_attributes
    FROM taxi_zones
""").fetchone()
check("Пропуски ключа справочника", lookup_profile[1], 0)
check("Повторяющиеся ключи справочника", lookup_profile[2], 0)
check("Пустые названия зон и районов", lookup_profile[3], 0)
show_table(["Строк справочника", "NULL ключа", "Повторяющихся ключей", "Пустых атрибутов"], [
    lookup_profile
])



join_profile = connection.execute("""
    SELECT count(*) AS rows_after_pickup_join,
           count(*) FILTER (WHERE pz.LocationID IS NULL) AS unmatched_pickup_rows
    FROM staging AS s
    LEFT JOIN taxi_zones AS pz ON s.PULocationID = pz.LocationID
""").fetchone()
check("Количество строк после JOIN зоны посадки", join_profile[0], staging_rows)
check("Посадки без соответствия до записи", join_profile[1], 0)

dropoff_join_profile = connection.execute("""
    SELECT count(*) AS rows_after_dropoff_join,
           count(*) FILTER (WHERE dz.LocationID IS NULL) AS unmatched_dropoff_rows
    FROM staging AS s
    LEFT JOIN taxi_zones AS pz ON s.PULocationID = pz.LocationID
    LEFT JOIN taxi_zones AS dz ON s.DOLocationID = dz.LocationID
""").fetchone()
check("Количество строк после JOIN зоны высадки", dropoff_join_profile[0], join_profile[0])
check("Высадки без соответствия до записи", dropoff_join_profile[1], 0)
show_table(["Этап", "Строк до", "Строк после", "Без соответствия"], [
    ("Зона посадки", staging_rows, *join_profile),
    ("Зона высадки", join_profile[0], *dropoff_join_profile),
])


Поле справочника,Тип
Borough,VARCHAR
LocationID,BIGINT
Zone,VARCHAR
service_zone,VARCHAR


Строк справочника,NULL ключа,Повторяющихся ключей,Пустых атрибутов
265,0,0,0


Этап,Строк до,Строк после,Без соответствия
Зона посадки,48235,48235,0
Зона высадки,48235,48235,0


### Сборка широкой таблицы

In [6]:
connection.execute("""
    CREATE OR REPLACE TEMP TABLE trips AS
    SELECT
        s.lpep_pickup_datetime AS pickup_at,
        s.lpep_dropoff_datetime AS dropoff_at,
        s.PULocationID AS pickup_zone_id,
        pz.Zone AS pickup_zone_name,
        pz.Borough AS pickup_borough,
        s.DOLocationID AS dropoff_zone_id,
        dz.Zone AS dropoff_zone_name,
        dz.Borough AS dropoff_borough,
        CAST(s.lpep_pickup_datetime AS DATE) AS pickup_date,
        EXTRACT(hour FROM s.lpep_pickup_datetime) AS pickup_hour,
        EXTRACT(isodow FROM s.lpep_pickup_datetime) AS pickup_isodow,
        date_diff('second', s.lpep_pickup_datetime, s.lpep_dropoff_datetime)
            / 60.0 AS duration_minutes,
        s.trip_distance,
        s.total_amount,
        s.quality_negative_total_amount_is_correction_violated AS negative_total_flag
    FROM staging AS s
    LEFT JOIN taxi_zones AS pz ON s.PULocationID = pz.LocationID
    LEFT JOIN taxi_zones AS dz ON s.DOLocationID = dz.LocationID
""")
prepared_rows = connection.execute("SELECT count(*) FROM trips").fetchone()[0]
check("Сохранение строк при преобразовании", prepared_rows, staging_rows)
show_table(
    ["Посадка", "Зона посадки", "Зона высадки", "Мили", "Минуты", "USD", "Флаг"],
    connection.execute("""
        SELECT pickup_at, pickup_zone_name, dropoff_zone_name, trip_distance,
               duration_minutes, total_amount, negative_total_flag
        FROM trips
        ORDER BY pickup_at, dropoff_at, pickup_zone_id, dropoff_zone_id, total_amount
        LIMIT 5
    """).fetchall(),
)


Посадка,Зона посадки,Зона высадки,Мили,Минуты,USD,Флаг
2025-01-01 00:01:11,Forest Hills,Jamaica,3.9,11.683333333333334,20.9,False
2025-01-01 00:01:40,East Harlem North,Yorkville East,2.91,16.683333333333334,27.54,False
2025-01-01 00:03:01,East Harlem South,University Heights/Morris Heights,5.93,14.183333333333334,34.0,False
2025-01-01 00:04:05,Washington Heights North,Hamilton Heights,3.22,11.733333333333333,19.5,False
2025-01-01 00:05:19,Central Harlem,East Harlem North,1.05,5.766666666666667,12.48,False


## 4. Запись и повторное чтение S3

Используем способ ЛР № 3: удаляем только целевой объект `trips.parquet` и записываем
его заново. Повторный запуск заменяет результат, не добавляя копий строк.
Если запись прервётся после удаления, этот объект потребуется восстановить
повторным полным запуском. Остальные объекты S3 не затрагиваются.

После записи открываем новое соединение DuckDB и читаем именно объект S3.
Автоматическое добавление полей из каталогов отключено: проверяем физическую
схему файла; логическая дата уже явно задана его путём.

In [7]:
assert MART_PATH == f"s3://mart/{PREFIX}/trips.parquet"
assert MART_KEY == f"{PREFIX}/trips.parquet"
client.head_bucket(Bucket="mart")
client.delete_object(Bucket="mart", Key=MART_KEY)
connection.execute(f"""
    COPY trips TO {literal(MART_PATH)} (FORMAT PARQUET, COMPRESSION ZSTD)
""")

published = duckdb.connect()
configure_s3(published)
published.execute(f"""
    CREATE OR REPLACE TEMP VIEW published_trips AS
    SELECT * FROM read_parquet({literal(MART_PATH)}, hive_partitioning = false)
""")
output_head = client.head_object(Bucket="mart", Key=MART_KEY)
show_table(["Объект", "Размер, байт"], [(MART_PATH, output_head["ContentLength"])])

Объект,"Размер, байт"
s3://mart/green_tripdata/source_ingested_on=2026-09-30/trips.parquet,979314


<a id="checks"></a>
## 5. Проверки сохранённой витрины

Все запросы к `published_trips` ниже читают записанный Parquet на S3.
Ожидания следуют из модели и входного `staging`. При нарушении любого условия
`assert` останавливает выполнение. Сводка в конце показывает фактическое
и ожидаемое значение каждой проверки.

### Количество и схема

Все строки `staging` сохраняются. Проверяем нулевое число исключений, баланс
`staging = mart + исключено`, совпадение с подготовленной таблицей
и схему записанного файла.

In [8]:
mart_rows = published.execute("SELECT count(*) FROM published_trips").fetchone()[0]
excluded_rows = staging_rows - prepared_rows
check("Исключённые строки", excluded_rows, 0)
check("Баланс входных и сохранённых строк", staging_rows, mart_rows + excluded_rows)
check("Количество после чтения S3", mart_rows, prepared_rows)

expected_schema = [
    ("pickup_at", "TIMESTAMP"),
    ("dropoff_at", "TIMESTAMP"),
    ("pickup_zone_id", "INTEGER"),
    ("pickup_zone_name", "VARCHAR"),
    ("pickup_borough", "VARCHAR"),
    ("dropoff_zone_id", "INTEGER"),
    ("dropoff_zone_name", "VARCHAR"),
    ("dropoff_borough", "VARCHAR"),
    ("pickup_date", "DATE"),
    ("pickup_hour", "BIGINT"),
    ("pickup_isodow", "BIGINT"),
    ("duration_minutes", "DOUBLE"),
    ("trip_distance", "DOUBLE"),
    ("total_amount", "DOUBLE"),
    ("negative_total_flag", "BOOLEAN"),
]
actual_schema = [row[:2] for row in published.execute("DESCRIBE published_trips").fetchall()]
check("Имена, порядок и типы пятнадцати полей", actual_schema, expected_schema)
show_table(["Staging", "Подготовлено", "Прочитано из mart", "Исключено"], [
    (staging_rows, prepared_rows, mart_rows, excluded_rows)
])
show_table(["Поле", "Тип"], actual_schema)

Staging,Подготовлено,Прочитано из mart,Исключено
48235,48235,48235,0


Поле,Тип
pickup_at,TIMESTAMP
dropoff_at,TIMESTAMP
pickup_zone_id,INTEGER
pickup_zone_name,VARCHAR
pickup_borough,VARCHAR
dropoff_zone_id,INTEGER
dropoff_zone_name,VARCHAR
dropoff_borough,VARCHAR
pickup_date,DATE
pickup_hour,BIGINT


### Соответствия справочнику после чтения S3

Повторно читаем справочник в новом соединении и проверяем уже сохранённую
витрину для обеих ролей. Для каждой записи должны существовать коды зон посадки
и высадки, а их названия и районы
должны совпадать с исходным справочником. Проверяем также число строк каждого
контрольного соединения, чтобы исключить размножение при проверке.

In [9]:
published.execute(f"""
    CREATE OR REPLACE TEMP VIEW published_zones AS
    SELECT * FROM read_csv_auto(
        {literal(ZONES_PATH)}, header = true, hive_partitioning = false
    )
""")
saved_zone_profile = published.execute("""
    SELECT
        count(*) AS rows_after_join,
        count(*) FILTER (WHERE z.LocationID IS NULL) AS unmatched_rows,
        count(*) FILTER (
            WHERE p.pickup_zone_name IS DISTINCT FROM z.Zone
               OR p.pickup_borough IS DISTINCT FROM z.Borough
        ) AS mismatched_attributes
    FROM published_trips AS p
    LEFT JOIN published_zones AS z ON p.pickup_zone_id = z.LocationID
""").fetchone()
check("Количество строк при проверке зон посадки", saved_zone_profile[0], mart_rows)
check("Посадки без соответствия после записи", saved_zone_profile[1], 0)
check("Несовпадения атрибутов посадки", saved_zone_profile[2], 0)


saved_dropoff_zone_profile = published.execute("""
    SELECT
        count(*) AS rows_after_join,
        count(*) FILTER (WHERE z.LocationID IS NULL) AS unmatched_rows,
        count(*) FILTER (
            WHERE p.dropoff_zone_name IS DISTINCT FROM z.Zone
               OR p.dropoff_borough IS DISTINCT FROM z.Borough
        ) AS mismatched_attributes
    FROM published_trips AS p
    LEFT JOIN published_zones AS z ON p.dropoff_zone_id = z.LocationID
""").fetchone()
check("Количество строк при проверке зон высадки", saved_dropoff_zone_profile[0], mart_rows)
check("Высадки без соответствия после записи", saved_dropoff_zone_profile[1], 0)
check("Несовпадения атрибутов высадки", saved_dropoff_zone_profile[2], 0)
show_table(["Роль зоны", "Строк mart", "Строк после JOIN", "Без соответствия", "Несовпадений"], [
    ("Посадка", mart_rows, *saved_zone_profile),
    ("Высадка", mart_rows, *saved_dropoff_zone_profile),
])


Роль зоны,Строк mart,Строк после JOIN,Без соответствия,Несовпадений
Посадка,48235,48235,0,0
Высадка,48235,48235,0,0


### Обязательные поля и меры

Ключа поездки нет, поэтому проверка его уникальности неприменима. Проверяем
заполненность времён, кодов, названий и районов обеих зон, календарных признаков,
длительности и флага. Без них теряется смысл строки или необходимого среза.
Для `total_amount` и `trip_distance` допускаем исходные пропуски: новые
появляться не должны, их количество сравниваем со `staging`.

Длительность должна оставаться положительной по решению ЛР № 3. Верхнюю границу
в ЛР № 3 не задавали: максимум показываем для контроля и не вводим скрытого
отсечения. Отрицательный минимум суммы допустим при сохранённом флаге.

In [10]:
null_counts = published.execute("""
    SELECT
        count(*) FILTER (WHERE pickup_at IS NULL),
        count(*) FILTER (WHERE dropoff_at IS NULL),
        count(*) FILTER (WHERE pickup_zone_id IS NULL),
        count(*) FILTER (WHERE pickup_zone_name IS NULL),
        count(*) FILTER (WHERE pickup_borough IS NULL),
        count(*) FILTER (WHERE dropoff_zone_id IS NULL),
        count(*) FILTER (WHERE dropoff_zone_name IS NULL),
        count(*) FILTER (WHERE dropoff_borough IS NULL),
        count(*) FILTER (WHERE pickup_date IS NULL),
        count(*) FILTER (WHERE pickup_hour IS NULL),
        count(*) FILTER (WHERE pickup_isodow IS NULL),
        count(*) FILTER (WHERE duration_minutes IS NULL),
        count(*) FILTER (WHERE negative_total_flag IS NULL),
        count(*) FILTER (WHERE trip_distance IS NULL),
        count(*) FILTER (WHERE total_amount IS NULL)
    FROM published_trips
""").fetchone()
required_names = [
    name for name, _ in expected_schema if name not in {"trip_distance", "total_amount"}
]
for name, null_count in zip(required_names, null_counts[:-2], strict=True):
    check(f"NULL в обязательном поле {name}", null_count, 0)
check("NULL расстояния относительно staging", null_counts[-2], source_distance_profile[0])
check("NULL суммы относительно staging", null_counts[-1], source_null_amount)
show_table(["Поле", "NULL", "Ожидается"], [
    *[(name, value, 0) for name, value in zip(required_names, null_counts[:-2], strict=True)],
    ("trip_distance", null_counts[-2], source_distance_profile[0]),
    ("total_amount", null_counts[-1], source_null_amount),
])

measure_profile = published.execute("""
    SELECT min(duration_minutes), max(duration_minutes),
           min(total_amount), max(total_amount),
           count(*) FILTER (WHERE duration_minutes <= 0)
    FROM published_trips
""").fetchone()
source_amount_range = connection.execute("""
    SELECT min(total_amount), max(total_amount) FROM staging
""").fetchone()
check("Неположительные длительности", measure_profile[4], 0)
check("Минимум и максимум суммы сохранены", measure_profile[2:4], source_amount_range)
show_table(["Мера", "Минимум", "Максимум", "Единица"], [
    ("duration_minutes", measure_profile[0], measure_profile[1], "Минуты"),
    ("total_amount", measure_profile[2], measure_profile[3], "USD"),
])

Поле,NULL,Ожидается
pickup_at,0,0
dropoff_at,0,0
pickup_zone_id,0,0
pickup_zone_name,0,0
pickup_borough,0,0
dropoff_zone_id,0,0
dropoff_zone_name,0,0
dropoff_borough,0,0
pickup_date,0,0
pickup_hour,0,0


Мера,Минимум,Максимум,Единица
duration_minutes,0.016666666666666666,1439.65,Минуты
total_amount,-114.0,371.4,USD


<a id="distance-checks"></a>
### Расстояние: сохранение значений и правило будущего анализа

Сравниваем со `staging` пропуски, нули, отрицательные и бесконечные значения
или `NaN`, минимум и максимум. Все исходные расстояния должны сохраниться,
включая максимум 84 731,57 мили. Дополнительно сверяем количество строк
с нулём и с расстоянием больше 1000, а также размер будущей аналитической выборки.

В этом снимке 2 629 нулевых расстояний и 21 расстояние больше 1000.
Обе группы остаются в витрине для остальных показателей, а из дальнейшего
анализа связи расстояния с длительностью и стоимостью исключаются одинаково.
Запрос ниже только проверяет количества, строки витрины не удаляются.

In [11]:
distance_profile = published.execute("""
    SELECT
        count(*) FILTER (WHERE trip_distance IS NULL),
        count(*) FILTER (WHERE trip_distance = 0),
        count(*) FILTER (WHERE trip_distance < 0),
        count(*) FILTER (WHERE NOT isfinite(trip_distance)),
        min(trip_distance), max(trip_distance)
    FROM published_trips
""").fetchone()
check("Профиль расстояния совпадает со staging", distance_profile, source_distance_profile)
check("Все нулевые расстояния сохранены", distance_profile[1], source_distance_profile[1])
distance_policy_profile = published.execute(f"""
    SELECT
        count(*) FILTER (WHERE trip_distance = 0),
        count(*) FILTER (WHERE trip_distance > {MAX_DISTANCE_FOR_RELATIONSHIP_MILES}),
        count(*) FILTER (
            WHERE trip_distance = 0 OR trip_distance > {MAX_DISTANCE_FOR_RELATIONSHIP_MILES}
        ),
        count(*) FILTER (
            WHERE trip_distance > 0
              AND trip_distance <= {MAX_DISTANCE_FOR_RELATIONSHIP_MILES}
              AND isfinite(trip_distance)
        )
    FROM published_trips
""").fetchone()
check("Группы расстояний сохранены для будущего анализа", distance_policy_profile,
      source_distance_policy_profile)
show_table(["Слой", "NULL", "Нулей", "Отрицательных", "Бесконечность / NaN", "Мин., мили",
            "Макс., мили"], [
    ("Staging", *source_distance_profile),
    ("Mart", *distance_profile),
])
show_table(["Нули", "Больше 1000", "Не участвуют в анализе связи", "Подходящие расстояния"], [
    distance_policy_profile
])

Слой,NULL,Нулей,Отрицательных,Бесконечность / NaN,"Мин., мили","Макс., мили"
Staging,0,2629,0,0,0.0,84731.57
Mart,0,2629,0,0,0.0,84731.57


Нули,Больше 1000,Не участвуют в анализе связи,Подходящие расстояния
2629,21,2650,45585


### Календарные признаки, длительность и флаг

Проверяем диапазоны часа и ISO-дня, соответствие признаков временным меткам,
положительную длительность и январский период посадки. Сверяем флаг с самим
отрицательным значением и количество помеченных записей со `staging`.

In [12]:
derived_violations = published.execute("""
    SELECT count(*) FROM published_trips
    WHERE pickup_hour NOT BETWEEN 0 AND 23
       OR pickup_isodow NOT BETWEEN 1 AND 7
       OR pickup_date IS DISTINCT FROM CAST(pickup_at AS DATE)
       OR pickup_hour IS DISTINCT FROM EXTRACT(hour FROM pickup_at)
       OR pickup_isodow IS DISTINCT FROM EXTRACT(isodow FROM pickup_at)
       OR duration_minutes IS DISTINCT FROM date_diff('second', pickup_at, dropoff_at) / 60.0
       OR pickup_at < TIMESTAMP '2025-01-01'
       OR pickup_at >= TIMESTAMP '2025-02-01'
""").fetchone()[0]
flag_profile = published.execute("""
    SELECT
        count(*) FILTER (WHERE negative_total_flag) AS flagged_rows,
        count(*) FILTER (WHERE total_amount < 0) AS negative_rows,
        count(*) FILTER (
            WHERE negative_total_flag IS DISTINCT FROM coalesce(total_amount < 0, false)
        ) AS flag_mismatches
    FROM published_trips
""").fetchone()
check("Нарушения вычисленных полей и периода", derived_violations, 0)
check("Количество флагов относительно staging", flag_profile[0], source_flagged_rows)
check("Отрицательные суммы имеют флаг", flag_profile[1], source_flagged_rows)
check("Несоответствия суммы и флага", flag_profile[2], 0)
show_table(["Нарушения признаков", "Флаг TRUE", "Отрицательная сумма", "Несоответствия"], [
    (derived_violations, *flag_profile)
])

Нарушения признаков,Флаг TRUE,Отрицательная сумма,Несоответствия
0,152,152,0


### Сохранение значений и кратности строк

Бизнес-ключа поездки нет. Сверяем все пятнадцать полей записанной витрины
с ожидаемой проекцией всех строк `staging`, обогащённых справочником, через
`EXCEPT ALL` в обе стороны. `EXCEPT ALL` учитывает число совпадающих строк:
совпавшие аналитические проекции не должны исчезать. Проверяются также точный
перенос отрицательных сумм с флагами и расстояние, а также названия и районы обеих зон.

In [13]:
published.execute(f"""
    CREATE OR REPLACE TEMP VIEW expected_trips AS
    SELECT
        s.lpep_pickup_datetime AS pickup_at,
        s.lpep_dropoff_datetime AS dropoff_at,
        s.PULocationID AS pickup_zone_id,
        pz.Zone AS pickup_zone_name,
        pz.Borough AS pickup_borough,
        s.DOLocationID AS dropoff_zone_id,
        dz.Zone AS dropoff_zone_name,
        dz.Borough AS dropoff_borough,
        CAST(s.lpep_pickup_datetime AS DATE) AS pickup_date,
        EXTRACT(hour FROM s.lpep_pickup_datetime) AS pickup_hour,
        EXTRACT(isodow FROM s.lpep_pickup_datetime) AS pickup_isodow,
        date_diff('second', s.lpep_pickup_datetime, s.lpep_dropoff_datetime)
            / 60.0 AS duration_minutes,
        s.trip_distance,
        s.total_amount,
        s.quality_negative_total_amount_is_correction_violated AS negative_total_flag
    FROM read_parquet({literal(STAGING_PATH)}, hive_partitioning = false) AS s
    LEFT JOIN published_zones AS pz ON s.PULocationID = pz.LocationID
    LEFT JOIN published_zones AS dz ON s.DOLocationID = dz.LocationID
""")
content_difference = published.execute("""
    SELECT 'Отсутствуют в mart' AS direction, count(*) AS differing_rows
    FROM (SELECT * FROM expected_trips EXCEPT ALL SELECT * FROM published_trips)
    UNION ALL
    SELECT 'Лишние в mart', count(*)
    FROM (SELECT * FROM published_trips EXCEPT ALL SELECT * FROM expected_trips)
""").fetchall()
for direction, differing_rows in content_difference:
    check(direction, differing_rows, 0)
show_table(["Направление сверки", "Различающихся строк"], content_difference)

Направление сверки,Различающихся строк
Отсутствуют в mart,0
Лишние в mart,0


### Пригодность для вопросов проекта

Один контрольный запрос группирует записи по коду и названию зоны посадки,
а также району. Показываем первые десять групп; баланс считаем по всем группам.
Сумма их чисел записей должна равняться общему числу строк. Запрос подтверждает
возможность работы с географическим срезом после обогащения; полный расчёт
медиан, средних и содержательные выводы остаются следующему этапу.

Для первого уточняющего вопроса проверены наличие и точный перенос расстояния,
длительности и суммы с правилами их будущего использования. Для второго - коды,
названия и районы зон посадки и высадки, включая покрытие обоих соединений.
Группировка пар зон и анализ связи мер остаются следующему этапу.


In [14]:
zone_counts = published.execute("""
    SELECT pickup_zone_id, pickup_zone_name, pickup_borough, count(*) AS records_count
    FROM published_trips
    GROUP BY pickup_zone_id, pickup_zone_name, pickup_borough
    ORDER BY records_count DESC, pickup_zone_id
""").fetchall()
check("Баланс контрольной группировки", sum(row[3] for row in zone_counts), mart_rows)
show_table(["Код зоны", "Зона посадки", "Район", "Число записей"], zone_counts[:10])
show_table(["Всего групп", "Сумма записей по всем группам"], [
    (len(zone_counts), sum(row[3] for row in zone_counts))
])

Код зоны,Зона посадки,Район,Число записей
74,East Harlem North,Manhattan,11672
75,East Harlem South,Manhattan,8060
43,Central Park,Manhattan,2776
95,Forest Hills,Queens,2236
166,Morningside Heights,Manhattan,2126
97,Fort Greene,Brooklyn,1993
41,Central Harlem,Manhattan,1974
82,Elmhurst,Queens,1798
65,Downtown Brooklyn/MetroTech,Brooklyn,1508
130,Jamaica,Queens,1324


Всего групп,Сумма записей по всем группам
212,48235


### Контрольная сводка

Дополнительно проверяем, что метаданные обоих входных объектов не изменились за время
выполнения. Все проверки относятся к конкретному снимку, а не доказывают
полноту или достоверность сведений о реальных поездках.

In [15]:
source_after = client.head_object(Bucket="staging", Key=f"{PREFIX}/staging.parquet")
check("Входной объект не изменён", (
    source_after["ETag"], source_after["ContentLength"], source_after["LastModified"]
), (
    source_before["ETag"], source_before["ContentLength"], source_before["LastModified"]
))
zones_after = client.head_object(Bucket="raw", Key=ZONES_KEY)
check("Объект справочника не изменён", (
    zones_after["ETag"], zones_after["ContentLength"], zones_after["LastModified"]
), (
    zones_before["ETag"], zones_before["ContentLength"], zones_before["LastModified"]
))
show_table(["Проверка", "Получено", "Ожидается", "Итог"], [
    (name, actual, expected, "Пройдена" if passed else "Нарушение")
    for name, actual, expected, passed in check_results
])

# Контрольная сводка выполненного ноутбука.
run_summary = {
    "staging_rows": staging_rows,
    "prepared_rows": prepared_rows,
    "mart_rows": mart_rows,
    "excluded_rows": excluded_rows,
    "schema": actual_schema,
    "null_counts": null_counts,
    "measure_profile": measure_profile,
    "distance_profile": distance_profile,
    "source_distance_profile": source_distance_profile,
    "distance_policy_profile": distance_policy_profile,
    "derived_violations": derived_violations,
    "flag_profile": flag_profile,
    "content_difference": content_difference,
    "lookup_profile": lookup_profile,
    "join_profile": join_profile,
    "dropoff_join_profile": dropoff_join_profile,
    "saved_zone_profile": saved_zone_profile,
    "saved_dropoff_zone_profile": saved_dropoff_zone_profile,
    "zone_counts": zone_counts,
    "checks_passed": sum(passed for _, _, _, passed in check_results),
    "checks_total": len(check_results),
    "source_etag": source_after["ETag"],
    "source_size": source_after["ContentLength"],
    "zones_etag": zones_after["ETag"],
    "zones_size": zones_after["ContentLength"],
}
show_table(["Проверок", "Пройдено", "Строк mart", "Сохранено флагов"], [
    (len(check_results), run_summary["checks_passed"], mart_rows, flag_profile[0])
])
published.close()
connection.close()

Проверка,Получено,Ожидается,Итог
Наличие нужных полей staging,set(),set(),Пройдена
Критические нарушения ЛР № 3 на входе,0,0,Пройдена
Наличие полей справочника,set(),set(),Пройдена
Пропуски ключа справочника,0,0,Пройдена
Повторяющиеся ключи справочника,0,0,Пройдена
Пустые названия зон и районов,0,0,Пройдена
Количество строк после JOIN зоны посадки,48235,48235,Пройдена
Посадки без соответствия до записи,0,0,Пройдена
Количество строк после JOIN зоны высадки,48235,48235,Пройдена
Высадки без соответствия до записи,0,0,Пройдена


Проверок,Пройдено,Строк mart,Сохранено флагов
50,50,48235,152


<a id="result"></a>
## 6. Оформите результат

Модель, происхождение всех выходных полей, краткий итог проверок и ограничения
описаны в [docs/mart.md](../docs/mart.md). Демонстрационная витрина поддерживает
основной и два уточняющих вопроса проекта. Для студенческой работы достаточно
одного выбранного вопроса из принятого `Project Brief`.

Итог сборки: все 48 235 строк `staging` сохранены в витрине, исключённых строк нет.
Гранулярность и кратность строк сохранены; оба соединения со справочником
не изменили количество записей. В выходном файле 15 полей, сохранены 152 флага
отрицательной суммы. Все 50 проверок сборки пройдены.

Витрина сохранена и прочитана из S3:

```text
s3://mart/green_tripdata/source_ingested_on=2026-09-30/trips.parquet
```

Для дальнейшего анализа зафиксировано общее правило: 2 629 строк с нулевым
расстоянием и 21 строка с расстоянием больше 1000 остаются для остальных
показателей, но исключаются из анализа связи расстояния с длительностью
и стоимостью. При сравнении со стоимостью также требуется заполненная
неотрицательная сумма. Это условие будущего расчёта, а не фильтр сборки витрины.

Ограничения: бизнес-ключа поездки нет; соответствие справочнику не доказывает
точность географии, а его снимок не описывает историю изменения зон.
Отрицательный флаг обозначает знак суммы, причина неизвестна. Длительности
от одной секунды до почти суток сохранены по правилам ЛР № 3 и требуют
осторожности при интерпретации. Полный расчёт метрик и содержательные выводы
остаются следующему этапу.

Ноутбук сохранён с актуальными SQL-запросами и результатами выполнения.
Подробные проверки и таблицы результатов находятся в [разделе 5](#checks);
в описании модели оставлен краткий итог.